## QAT INT8 TensorRT Inference (Test Set)

Evaluate QAT INT8 checkpoints via TensorRT on the test set (imagenet2), across all seeds.
Pipeline: load QAT checkpoint → export QDQ ONNX → build TRT engine → run inference.

The Q/DQ nodes baked into the ONNX give TRT the quantization scales directly — no runtime calibration needed.

In [ ]:
SKIP_EXISTING = True
TEST_IMAGENET = "/home/pf4636/imagenet2"
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/test/qat"

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [ ]:
import json
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import onnx
import modelopt.torch.opt as mto

from dataclasses import replace
from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_imagenet_dataset
from trt.trt_builder import build_engine
from trt.trt_infer import trt_evaluate
from quantize import get_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
QAT_CKPT_ROOT  = PROJECT_ROOT / ".checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / ".checkpoints"
ONNX_DIR       = PROJECT_ROOT / "onnx" / "qat"
ENGINE_DIR     = PROJECT_ROOT / "engines" / "qat"
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_BITS     = [8, 4, 2, 1]

checkpoints = {}
skipped = []
for bits in INPUT_BITS:
    run_name = f"int8_in{bits}b"
    run_dir = QAT_CKPT_ROOT / run_name
    if not run_dir.exists():
        skipped.append(f"int8/in{bits}b (no dir)")
        continue
    for seed_dir in sorted(run_dir.iterdir()):
        if not seed_dir.is_dir() or not seed_dir.name.startswith("seed_"):
            continue
        weights = seed_dir / "qat_modelopt_best.pth"
        mostate = seed_dir / "qat_modelopt_best_mostate.pt"
        seed_num = int(seed_dir.name.split("_")[1])
        fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / seed_dir.name / "best.pth"
        if not weights.exists() or not mostate.exists() or not fp32_ckpt.exists():
            skipped.append(f"int8/in{bits}b/{seed_dir.name}")
            continue
        checkpoints[(bits, seed_dir.name)] = {
            "weights": weights,
            "mostate": mostate,
            "fp32_ckpt": fp32_ckpt,
            "seed": seed_num,
        }

print(f"QAT INT8 checkpoints found: {len(checkpoints)}")
for (bits, seed), paths in checkpoints.items():
    print(f"  in{bits}b / {seed}")
if skipped:
    print(f"\nSkipped (not complete): {', '.join(skipped)}")
print(f"Test set: {TEST_IMAGENET}")

In [ ]:
def load_qat_model(bits: int, seed_name: str) -> nn.Module:
    paths = checkpoints[(bits, seed_name)]
    model = get_model(str(paths["fp32_ckpt"]), num_classes=100)
    restore_modelopt_state(model, str(paths["mostate"]))
    ckpt = torch.load(paths["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model

def export_qat_onnx(model: nn.Module, onnx_path: Path):
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    model.eval().cpu()
    dummy = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        torch.onnx.export(
            model, (dummy,), str(onnx_path),
            input_names=["images"],
            output_names=["logits"],
            opset_version=17,
            dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
            export_params=True,
            do_constant_folding=True,
            dynamo=False,
        )
    print(f"    ONNX saved ({onnx_path.stat().st_size / 1e6:.1f} MB)")

def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_IMAGENET)
    dataset = build_imagenet_dataset(test_cfg, split="val")
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

## Export QDQ ONNX

The Q/DQ nodes from ModelOpt's fake-quant graph are preserved in the ONNX export, so TRT gets the scales directly.

In [ ]:
for (bits, seed_name), paths in checkpoints.items():
    onnx_path = ONNX_DIR / f"int8_in{bits}b" / seed_name / "resnet18_qat_int8_qdq.onnx"

    if SKIP_EXISTING and onnx_path.exists():
        print(f"  int8/in{bits}b/{seed_name}: ONNX exists, skipping")
        continue

    print(f"\n  Exporting int8/in{bits}b/{seed_name} ...")
    model = load_qat_model(bits, seed_name)
    export_qat_onnx(model, onnx_path)
    del model
    torch.cuda.empty_cache()

print("\nAll QAT ONNX exports ready.")

In [ ]:
print("Q/DQ node counts:")
for (bits, seed_name) in checkpoints:
    path = ONNX_DIR / f"int8_in{bits}b" / seed_name / "resnet18_qat_int8_qdq.onnx"
    if not path.exists():
        print(f"  int8/in{bits}b/{seed_name}: NOT FOUND")
        continue
    m = onnx.load(str(path), load_external_data=False)
    ops = [n.op_type for n in m.graph.node]
    print(f"  int8/in{bits}b/{seed_name}: Q={ops.count('QuantizeLinear'):3d}  DQ={ops.count('DequantizeLinear'):3d}  size={path.stat().st_size/1e6:.1f}MB")

## Build TRT Engines & Run Inference

In [ ]:
OUT_DIR = Path(OUTPUT_ROOT).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

all_records = []
criterion = nn.CrossEntropyLoss()

for (bits, seed_name), paths in checkpoints.items():
    seed_num = paths["seed"]
    run_id = f"trt_qat_int8_b{bits}_cuda"
    result_path = OUT_DIR / seed_name / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  int8/in{bits}b/{seed_name}: exists, skipping")
        with open(result_path) as f:
            all_records.append(json.load(f))
        continue

    print(f"\n--- QAT INT8 TRT, input_bits={bits}, {seed_name} ---")

    onnx_path = ONNX_DIR / f"int8_in{bits}b" / seed_name / "resnet18_qat_int8_qdq.onnx"
    engine_path = ENGINE_DIR / f"int8_in{bits}b" / seed_name / f"{run_id}.engine"

    if not engine_path.exists():
        print(f"  Building TRT engine from {onnx_path.name} ...")
        engine_path.parent.mkdir(parents=True, exist_ok=True)
        build_engine(
            onnx_path=onnx_path,
            engine_path=engine_path,
            precision="int8",
            batch_size=1,
            workspace_mb=2048,
        )
    else:
        print(f"  Engine exists: {engine_path.name}")

    cfg = ExperimentConfig(
        backend="tensorrt",
        device="cuda",
        batch_size=1,
        seed=seed_num,
        num_eval_batches=None,
        input_quant_bits=bits,
        model_precision="int8",
    )
    set_seed(cfg)

    test_loader = build_test_loader(cfg)

    t0 = time.perf_counter()
    tracker = trt_evaluate(engine_path, cfg, test_loader, criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.3f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {
            "qat_precision": "int8",
            "input_quant_bits": bits,
            "seed": seed_num,
            "backend": "tensorrt",
        },
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    result_path.parent.mkdir(parents=True, exist_ok=True)
    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    all_records.append(payload)
    torch.cuda.empty_cache()

print(f"\n{len(all_records)} runs complete.")

## Per-Run Results

In [ ]:
rows = []
for rec in all_records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    seed = extra.get("seed", 42)
    rows.append({
        "method": "qat_int8_trt",
        "input_bits": bits,
        "seed": seed,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "lat_std": r.get("infer_ms_std", None),
        "throughput": r.get("throughput_infer_sps", None),
    })

df_all = pd.DataFrame(rows).sort_values(
    ["input_bits", "seed"], ascending=[True, True]
).reset_index(drop=True)
df_all

## Averaged Results (mean ± std across seeds)

In [ ]:
avg_df = df_all.groupby(["method", "input_bits"]).agg(
    top1_mean=("top1", "mean"),
    top1_std=("top1", "std"),
    top5_mean=("top5", "mean"),
    top5_std=("top5", "std"),
    infer_ms_mean=("lat_ms", "mean"),
    infer_ms_std=("lat_ms", "std"),
    throughput_mean=("throughput", "mean"),
    n_seeds=("seed", "count"),
).round(3).reset_index()

avg_df = avg_df.sort_values("input_bits").reset_index(drop=True)
avg_df

## Compare: QAT TRT vs QAT PyTorch

In [ ]:
qat_pytorch_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_torch_avg_test.json"
if qat_pytorch_path.exists():
    df_pytorch = pd.read_json(qat_pytorch_path)
    df_pt_int8 = df_pytorch[df_pytorch["qat_precision"] == "qat_int8"].rename(columns={
        "top1_mean": "pt_top1", "top5_mean": "pt_top5",
        "infer_ms_mean": "pt_lat_ms", "throughput_mean": "pt_throughput",
    })[["input_bits", "pt_top1", "pt_top5", "pt_lat_ms"]]

    comparison = avg_df.merge(df_pt_int8, on="input_bits", how="left")
    comparison["top1_delta"] = comparison["top1_mean"] - comparison["pt_top1"]
    comparison["speedup"] = comparison["pt_lat_ms"] / comparison["infer_ms_mean"]
    comparison
else:
    print("No QAT PyTorch results found — run 03_2_qat_test.ipynb first.")

## Save Results

In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_trt_avg_test.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")